# Chapter 11: Covering Maps

**Source Span:** John M. Lee, *Introduction to Topological Manifolds*, Second Edition, Chapter 11, printed pp. 277-306, PDF pp. 295-324.

## Chapter Goal

This notebook turns the theory of covering maps into inspectable geometry and finite algebra. By the end, a covering map should feel less like a formal definition and more like a machine with four visible parts: local sheets stacked over evenly covered neighborhoods, lifts of paths that move between sheets, monodromy permutations of each fiber, and a universal cover that stores homotopy classes of paths as actual points. The chapter also explains why those pictures are not merely suggestive. The endpoint of a lifted path is invariant under path homotopy, the subgroup induced by a covering is exactly the stabilizer of a fiber point, and the algebraic containment of fundamental-group images decides whether a map lifts.

The source chapter moves from local definitions to global classification tools. We follow that order but make each transition computational: first a local chart for the exponential cover, then a failure example showing why a local homeomorphism can miss the covering condition, then a degree-divisibility test for lifting maps through finite circle covers, then monodromy as a finite permutation action, and finally the lattice model of the universal cover of the torus. The examples are deliberately small. Covering-space theory works because tiny evenly covered neighborhoods force rigid global behavior, so the most useful computations are often simple invariants checked in many places.

The prose, diagrams, code, and checks here are standalone. The PDF was used only to orient terminology, theorem order, and examples from the assigned source span; no textbook text, figures, screenshots, crops, or exercise statements are reproduced.


In [ ]:
# geometry-setup:v1
# Machine-managed by scripts/update_notebook_setup.py. Do not edit this cell by hand.

from __future__ import annotations

import json as _geometry_json
import os as _geometry_os
from pathlib import Path as _GeometryPath
import sys as _geometry_sys

GEOMETRY_SETUP = _geometry_json.loads(
    r"""
{
  "colab_url": "https://colab.research.google.com/github/Rah-Rah-Mitra/Geometry/blob/main/Introduction-to-Topological-Manifolds/chapter-11-covering-maps/11-covering-maps.ipynb",
  "course_dir": "Introduction-to-Topological-Manifolds",
  "course_title": "Introduction to Topological Manifolds",
  "github_url": "https://github.com/Rah-Rah-Mitra/Geometry/blob/main/Introduction-to-Topological-Manifolds/chapter-11-covering-maps/11-covering-maps.ipynb",
  "jupyterlite": false,
  "marker": "geometry-setup:v1",
  "notebook_kind": "lesson",
  "notebook_path": "Introduction-to-Topological-Manifolds/chapter-11-covering-maps/11-covering-maps.ipynb",
  "notebook_title": "Chapter 11: Covering Maps",
  "repository": {
    "branch": "main",
    "name": "Geometry",
    "owner": "Rah-Rah-Mitra",
    "source_url": "https://github.com/Rah-Rah-Mitra/Geometry"
  },
  "requirements": "requirements/topology.txt",
  "runtime_profile": "topology"
}
"""
)


def _geometry_is_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _geometry_is_jupyterlite():
    return _geometry_sys.platform == "emscripten" or "pyodide" in _geometry_sys.modules


def _geometry_add_path(path):
    text = str(path)
    if text not in _geometry_sys.path:
        _geometry_sys.path.insert(0, text)


def _geometry_find_repo_root():
    candidates = []
    env_root = _geometry_os.environ.get("GEOMETRY_REPO_ROOT")
    if env_root:
        candidates.append(_GeometryPath(env_root).expanduser())
    candidates.append(_GeometryPath.cwd())
    for start in candidates:
        start = start.resolve()
        for current in (start, *start.parents):
            if (current / "course-manifest.json").exists() and (
                current / "metadata" / "runtime_profiles.yml"
            ).exists():
                return current
    raise RuntimeError(
        "Could not find the Geometry repository root. Start JupyterLab inside the "
        "Geometry checkout or set GEOMETRY_REPO_ROOT."
    )


def _geometry_run(command):
    import subprocess as _geometry_subprocess

    printable = " ".join(str(part) for part in command)
    print(f"+ {printable}")
    _geometry_subprocess.check_call([str(part) for part in command])


def _geometry_requirement_names(requirements_path, seen=None):
    seen = set() if seen is None else seen
    requirements_path = requirements_path.resolve()
    if requirements_path in seen or not requirements_path.exists():
        return []
    seen.add(requirements_path)
    names = []
    for raw_line in requirements_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        if line.startswith(("-r ", "--requirement ")):
            _, nested = line.split(maxsplit=1)
            names.extend(_geometry_requirement_names(requirements_path.parent / nested, seen))
            continue
        if line.startswith("-"):
            continue
        name = line
        for separator in ("==", ">=", "<=", "~=", "!=", ">", "<", ";"):
            name = name.split(separator, 1)[0]
        name = name.split("[", 1)[0].strip()
        if name:
            names.append(name)
    return sorted(set(names))


def _geometry_missing_requirements(requirements_path):
    import importlib.metadata as _geometry_metadata

    missing = []
    for name in _geometry_requirement_names(requirements_path):
        try:
            _geometry_metadata.distribution(name)
        except _geometry_metadata.PackageNotFoundError:
            missing.append(name)
    return missing


def _geometry_configured_roots(repo_root):
    course_dir = GEOMETRY_SETUP.get("course_dir")
    course_root = repo_root / course_dir if course_dir else repo_root
    return repo_root, course_root


if _geometry_is_jupyterlite():
    if not GEOMETRY_SETUP["jupyterlite"]:
        raise RuntimeError(
            "This Geometry notebook uses runtime profile "
            f"{GEOMETRY_SETUP['runtime_profile']!r}, which is not enabled for "
            "JupyterLite in course-manifest.json. Open it in Colab or local JupyterLab."
        )
    GEOMETRY_REPO_ROOT = _GeometryPath.cwd()
    GEOMETRY_COURSE_ROOT = (
        GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["course_dir"]
        if GEOMETRY_SETUP.get("course_dir")
        else GEOMETRY_REPO_ROOT
    )
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    if GEOMETRY_COURSE_ROOT.exists():
        _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        "Geometry setup: JupyterLite/Pyodide detected; shell, git, and pip steps "
        "were skipped."
    )
elif _geometry_is_colab():
    repository = GEOMETRY_SETUP["repository"]
    repo_url = repository["source_url"].rstrip("/") + ".git"
    branch = repository["branch"]
    GEOMETRY_REPO_ROOT = _GeometryPath(
        _geometry_os.environ.get("GEOMETRY_REPO_ROOT", "/content/Geometry")
    )
    sparse_paths = ["requirements", "metadata", "scripts", "course-manifest.json", "index.ipynb"]
    if GEOMETRY_SETUP.get("course_dir"):
        sparse_paths.append(GEOMETRY_SETUP["course_dir"])
    if not (GEOMETRY_REPO_ROOT / ".git").exists():
        if GEOMETRY_REPO_ROOT.exists() and any(GEOMETRY_REPO_ROOT.iterdir()):
            raise RuntimeError(
                f"{GEOMETRY_REPO_ROOT} exists but is not a git checkout. "
                "Set GEOMETRY_REPO_ROOT to an empty path or remove the directory."
            )
        _geometry_run(
            [
                "git",
                "clone",
                "--filter=blob:none",
                "--no-checkout",
                "--branch",
                branch,
                repo_url,
                GEOMETRY_REPO_ROOT,
            ]
        )
        _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "init", "--cone"])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "set", *sparse_paths])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "checkout", branch])
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-q", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: Colab ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )
else:
    GEOMETRY_REPO_ROOT = _geometry_find_repo_root()
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    missing = _geometry_missing_requirements(requirements_path)
    skip_install = _geometry_os.environ.get("GEOMETRY_SKIP_INSTALL") == "1"
    if missing and skip_install:
        print(
            "Geometry setup: GEOMETRY_SKIP_INSTALL=1, so missing profile packages "
            f"were not installed: {', '.join(missing)}"
        )
    elif missing:
        print(
            "Geometry setup: installing missing profile packages from "
            f"{requirements_path.relative_to(GEOMETRY_REPO_ROOT)}: {', '.join(missing)}"
        )
        _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: local checkout ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )


## Computational Translation Guide

| Topological object | Computational model in this notebook | What to inspect |
| --- | --- | --- |
| Covering map `q:E -> X` | A projection whose local preimage splits into disjoint components | Each sheet maps one-to-one onto the same base neighborhood. |
| Evenly covered neighborhood | A small arc in `S^1` and its lifted intervals in `R` | The intervals are disjoint, equal-width, and indexed by integer deck shifts. |
| Local section | A choice of one sheet over a base neighborhood | Picking a sheet gives a local inverse to `q`. |
| Surjective local homeomorphism that is not a cover | The restricted exponential map `(0,2) -> S^1` | Near `1` in the circle, two endpoint pieces cover only half of the chosen arc. |
| Lift of a path | An endpoint calculation in a chosen sheet | A loop of degree `m` moves sheet `j` to `j + m mod n` in the `n`-fold circle cover. |
| Lifting criterion | Subgroup containment | A degree `m` circle map lifts through the `n`th-power cover exactly when `m Z` is contained in `n Z`, equivalently `n` divides `m`. |
| Monodromy action | A right action of `pi_1(X,x)` on `q^{-1}(x)` | The generator of `pi_1(S^1)` acts as a cyclic permutation of the fiber. |
| Induced subgroup | Isotropy group of a fiber point | Loops that lift to loops at a selected sheet are exactly the stabilizer. |
| Normal covering | A covering whose induced subgroup is normal, or whose monodromy stabilizers match across the fiber | For abelian circle examples every subgroup is normal; nonabelian behavior appears through conjugacy. |
| Universal cover | A simply connected covering space | `R^2 -> T^2` is drawn as a lattice of copies of one torus chart, with fiber points differing by integer translations. |

## Library Routing

| Concept | Representation | Library choice | Reason |
| --- | --- | --- | --- |
| Evenly covered sheets and the failed local homeomorphism | Durable static schematic diagrams | `matplotlib` | These are planar incidence and interval diagrams; static PNGs are stable and easy to audit. |
| Lifting criterion for finite circle covers | Divisibility table and heat map | `pandas`, `sympy`, `matplotlib` | The criterion is algebraic subgroup containment, so exact integer divisibility is the right check; the heat map exposes the pattern. |
| Monodromy action on a finite fiber | Interactive directed cycle | `networkx`, `plotly` | The action is a graph/permutation object. Plotly preserves hover labels in a standalone HTML artifact. |
| Proof dependency structure | Directed theorem graph | `networkx`, `matplotlib` | The chapter is theorem-driven, so a DAG helps track which lifting facts support which classification facts. |
| Universal cover of the torus | Lattice of copies in `R^2` | `numpy`, `matplotlib` | Integer translations and modulo projection are numeric and visual at the same time. |

## Visual Storyboard

1. **Evenly covered sheets:** show a small arc in the base circle and several lifted intervals in the real line. The invariant is disjoint equal-width sheets.
2. **Local homeomorphism failure:** show the restricted exponential map from `(0,2)` and the incomplete endpoint sheets over a neighborhood of `1` in `S^1`. The invariant is that every component must cover the whole base neighborhood, and two components do not.
3. **Degree lifting criterion:** compute which degree maps `S^1 -> S^1` lift through the `n`-fold power cover. The invariant is `m % n == 0`.
4. **Monodromy action:** model a fiber of an `n`-sheeted circle cover as a cyclic `G`-set. The invariant is transitivity, with stabilizer `n Z`.
5. **Proof dependency scaffold:** draw the theorem dependencies from local sections and path lifting through monodromy, induced subgroups, and universal covers. The invariant is acyclicity of the dependency graph.
6. **Universal cover lattice:** display the universal cover of the torus as `R^2` tiled by integer translates. The invariant is that all translated fiber points have the same image modulo `Z^2`.
7. **Applied lab:** classify small circle-covering examples by induced subgroup, sheet count, and liftable degrees. The invariant is agreement between table entries and the subgroup containment test.


In [ ]:
from pathlib import Path
import json
import math
import sys

import numpy as np
import pandas as pd
import sympy as sp
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import plotly.graph_objects as go


def find_course_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'AGENTS.md').exists() and (candidate / 'source_map.json').exists() and (candidate / 'utils').exists():
            return candidate
        nested = candidate / 'Introduction-to-Topological-Manifolds'
        if (nested / 'AGENTS.md').exists() and (nested / 'source_map.json').exists() and (nested / 'utils').exists():
            return nested
    raise RuntimeError('Could not locate the Introduction-to-Topological-Manifolds course root')


BOOK_ROOT = find_course_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, chapter_artifact_root, display_artifact, save_csv, save_json, save_matplotlib, save_plotly_html

UNIT_KEY = 'chapter-11-covering-maps'
ARTIFACT_ROOT = chapter_artifact_root(UNIT_KEY, BOOK_ROOT)
FIGURES = ARTIFACT_ROOT / 'figures'
HTML = ARTIFACT_ROOT / 'html'
CHECKS = ARTIFACT_ROOT / 'checks'
TABLES = ARTIFACT_ROOT / 'tables'


def rel(path: Path) -> str:
    return Path(path).relative_to(BOOK_ROOT).as_posix()


print('Course root:', BOOK_ROOT.name)
print('Artifact root:', f'artifacts/{UNIT_KEY}')


In [ ]:
visual_storyboard = [
    {
        'id': 'evenly-covered-sheets',
        'concept': 'Covering maps are locally a stack of homeomorphic sheets.',
        'representation': 'Small base arc in S^1 with lifted intervals in R.',
        'library': 'matplotlib',
        'artifact': 'figures/evenly-covered-sheets.png',
        'inspection_target': 'Each sheet is an open interval mapped homeomorphically onto the same arc.',
        'validation': 'Lifted intervals are disjoint and have equal local-coordinate width.',
    },
    {
        'id': 'local-homeomorphism-not-covering',
        'concept': 'A surjective local homeomorphism can fail the covering condition.',
        'representation': 'Restricted exponential map (0,2) -> S^1 near the point 1.',
        'library': 'matplotlib',
        'artifact': 'figures/local-homeomorphism-not-covering.png',
        'inspection_target': 'Endpoint components cover only half of the selected base neighborhood.',
        'validation': 'Exactly two preimage components are incomplete over the chosen arc.',
    },
    {
        'id': 'lifting-criterion-degree-divisibility',
        'concept': 'The lifting criterion becomes subgroup containment for circle maps.',
        'representation': 'Divisibility heat map for degrees m through n-sheet covers.',
        'library': 'sympy, pandas, matplotlib',
        'artifact': 'figures/lifting-criterion-degree-divisibility.png',
        'inspection_target': 'Rows light up exactly at multiples of n.',
        'validation': 'Boolean lift result equals m % n == 0 for every table entry.',
    },
    {
        'id': 'monodromy-cycle-action',
        'concept': 'Monodromy is the action of pi_1 of the base on a fiber.',
        'representation': 'Directed cyclic permutation of a finite fiber.',
        'library': 'networkx, plotly',
        'artifact': 'html/monodromy-cycle-action.html',
        'inspection_target': 'The generator advances each sheet by one step.',
        'validation': 'The action graph has one orbit and generator order n.',
    },
    {
        'id': 'proof-dependency-scaffold',
        'concept': 'The chapter proof chain builds global algebra from local lifting.',
        'representation': 'Directed dependency graph of definitions and theorems.',
        'library': 'networkx, matplotlib',
        'artifact': 'figures/proof-dependency-covering-maps.png',
        'inspection_target': 'Lifting facts feed monodromy; monodromy feeds classification and universality.',
        'validation': 'Dependency graph is acyclic.',
    },
    {
        'id': 'universal-cover-torus-lattice',
        'concept': 'A universal cover is a simply connected space over the base.',
        'representation': 'Integer lattice copies of a torus chart in R^2.',
        'library': 'numpy, matplotlib',
        'artifact': 'figures/universal-cover-torus-lattice.png',
        'inspection_target': 'All marked fiber points differ by integer translations.',
        'validation': 'Modulo-Z^2 projection residual is zero within tolerance.',
    },
]

storyboard_path = save_json(
    {
        'chapter': 'Chapter 11: Covering Maps',
        'source_span': 'printed pp. 277-306, PDF pp. 295-324',
        'items': visual_storyboard,
    },
    CHECKS / 'visual-storyboard.json',
)
display_artifact(storyboard_path)


## Definitions and Basic Properties

A covering map is locally simple and globally interesting. The local condition says that every point of the base has a neighborhood whose full preimage is a disjoint union of open pieces, with each piece mapped homeomorphically onto the original neighborhood. Those pieces are the sheets. This is stronger than being a local homeomorphism because it requires a whole neighborhood in the base to be covered uniformly by complete sheets, not by partial fragments.

The exponential map `q(t) = exp(2 pi i t)` is the guiding model. Around a small arc in the circle, the preimage in `R` is a family of equally shaped intervals centered at the integers. Choosing one interval is choosing a local inverse, hence a local section. The same picture also explains why the number of sheets is constant over a connected base: over an evenly covered neighborhood, every sheet contains exactly one point over each base point, so nearby fibers have the same size. Connectedness then propagates that count across the base.

The second diagram below is the warning example. Restrict the same exponential formula to the open interval `(0,2)`. The map is still surjective and locally a homeomorphism, but near the base point `1` the preimage has two endpoint components that cover only one side of the base arc. Since every component over an evenly covered neighborhood must map onto the entire neighborhood, this cannot be a covering map. The failure is not local injectivity; it is missing complete sheets.


In [ ]:
arc_half_width = 0.16
sheet_centers = np.arange(-2, 3)
intervals = [(float(c - arc_half_width), float(c + arc_half_width)) for c in sheet_centers]
interval_lengths = [b - a for a, b in intervals]
sheets_disjoint = all(intervals[i][1] < intervals[i + 1][0] for i in range(len(intervals) - 1))
equal_widths = max(interval_lengths) - min(interval_lengths) < 1e-12

evenly_png = FIGURES / 'evenly-covered-sheets.png'
fig, axes = plt.subplots(2, 1, figsize=(9, 6), gridspec_kw={'height_ratios': [1.1, 1.0]})
ax = axes[0]
ax.axhline(0, color='0.25', lw=1.2)
colors = plt.cm.Set2(np.linspace(0.05, 0.9, len(intervals)))
for (a, b), c, color in zip(intervals, sheet_centers, colors):
    ax.plot([a, b], [0, 0], color=color, lw=10, solid_capstyle='round')
    ax.scatter([c], [0], s=60, color='black', zorder=3)
    ax.text(c, 0.18, f'sheet {int(c)}', ha='center', va='bottom', fontsize=9)
ax.set_xlim(-2.4, 2.4)
ax.set_ylim(-0.55, 0.55)
ax.set_yticks([])
ax.set_xlabel('covering coordinate t in R')
ax.set_title('q^{-1}(U) is a disjoint stack of intervals')

theta = np.linspace(-2 * np.pi * arc_half_width, 2 * np.pi * arc_half_width, 160)
ax = axes[1]
full = np.linspace(0, 2 * np.pi, 400)
ax.plot(np.cos(full), np.sin(full), color='0.75', lw=2)
ax.plot(np.cos(theta), np.sin(theta), color='#1f77b4', lw=7, solid_capstyle='round')
ax.scatter([1], [0], s=75, color='black', zorder=3)
ax.text(1.08, 0.05, 'base point 1', fontsize=9)
ax.text(0, 1.16, 'base arc U in S^1', ha='center', fontsize=10)
ax.set_aspect('equal')
ax.axis('off')
save_matplotlib(fig, evenly_png)
plt.close(fig)

delta = 0.16
components = [(0.0, delta), (1 - delta, 1 + delta), (2 - delta, 2.0)]
component_image_widths = [delta, 2 * delta, delta]
complete_components = [abs(width - 2 * delta) < 1e-12 for width in component_image_widths]
local_failure_png = FIGURES / 'local-homeomorphism-not-covering.png'
fig, ax = plt.subplots(figsize=(9, 2.8))
ax.plot([0, 2], [0, 0], color='0.25', lw=1.4)
ax.scatter([0, 2], [0, 0], s=90, facecolors='white', edgecolors='black', zorder=4)
for (a, b), complete, color in zip(components, complete_components, ['#e76f51', '#2a9d8f', '#e76f51']):
    ax.plot([a, b], [0, 0], color=color, lw=12, solid_capstyle='round')
    label = 'complete sheet' if complete else 'partial endpoint piece'
    ax.text((a + b) / 2, 0.18, label, ha='center', fontsize=9)
ax.text(1, -0.28, 'domain (0,2) with q(t)=exp(2 pi i t)', ha='center', fontsize=10)
ax.text(0.02, -0.12, '0 missing', ha='left', fontsize=8)
ax.text(1.98, -0.12, '2 missing', ha='right', fontsize=8)
ax.set_xlim(-0.12, 2.12)
ax.set_ylim(-0.45, 0.45)
ax.set_yticks([])
ax.set_xticks([0, 1, 2])
ax.set_title('A surjective local homeomorphism with incomplete sheets over 1 in S^1')
save_matplotlib(fig, local_failure_png)
plt.close(fig)

local_checks = {
    'evenly_covered_model': {
        'base_arc_half_width_turns': arc_half_width,
        'sheet_centers': [int(c) for c in sheet_centers],
        'sheet_count_sampled': int(len(sheet_centers)),
        'sheets_disjoint': bool(sheets_disjoint),
        'equal_local_widths': bool(equal_widths),
        'interval_lengths': [float(length) for length in interval_lengths],
    },
    'restricted_exponential_failure': {
        'domain': '(0,2)',
        'chosen_neighborhood_half_width_turns': delta,
        'component_image_widths_in_turns': [float(width) for width in component_image_widths],
        'complete_component_count': int(sum(complete_components)),
        'incomplete_component_count': int(len(complete_components) - sum(complete_components)),
    },
}
local_checks_path = save_json(local_checks, CHECKS / 'covering-local-sheet-checks.json')

assert sheets_disjoint
assert equal_widths
assert local_checks['restricted_exponential_failure']['incomplete_component_count'] == 2

display_artifact(evenly_png, width=760)
display_artifact(local_failure_png, width=760)
display_artifact(local_checks_path)


## Lifting Properties and the Lifting Criterion

The lifting theorems say that once the starting point in the fiber is fixed, a connected source cannot secretly choose two different lifts. In practice, this means lifted paths behave like deterministic state transitions. For a path in the base circle, the `n`-fold power covering `p_n:S^1 -> S^1` has a finite fiber with labels `0,1,...,n-1`. A base loop of degree `m` advances the label by `m` modulo `n`. If the lifted path starts and ends at the same fiber point, then the original loop lies in the subgroup induced by the covering.

The general lifting criterion turns this endpoint behavior into an algebraic test. Given `phi:Y -> X`, a lift through `q:E -> X` exists from a selected base point exactly when the subgroup `phi_*(pi_1(Y))` is contained in the subgroup `q_*(pi_1(E))`. The circle case makes the test visible. A degree `m` map from `S^1` to `S^1` sends the fundamental group to `m Z`. The `n`th-power covering induces `n Z`. Therefore the map lifts through `p_n` exactly when `m Z` is contained in `n Z`, which is the divisibility condition `n | m`.

The heat map below should be read row by row. Fix the covering degree `n`; the liftable maps are exactly the columns whose degree is a multiple of `n`. The second plot tracks one lifted loop with `n=5` and `m=7`: starting on sheet `0`, the endpoint sheet is `2`, because `7 mod 5 = 2`.


In [ ]:
degrees = list(range(-8, 9))
cover_degrees = list(range(2, 9))
criterion_rows = []
matrix = []
for n in cover_degrees:
    row = []
    for m in degrees:
        gcd_value = int(sp.gcd(abs(m), n))
        lifts = (m % n == 0)
        row.append(1 if lifts else 0)
        criterion_rows.append(
            {
                'cover_degree_n': n,
                'map_degree_m': m,
                'gcd_abs_m_n': gcd_value,
                'mZ_contained_in_nZ': lifts,
                'lift_exists': lifts,
            }
        )
    matrix.append(row)
matrix = np.array(matrix)

criterion_table_path = save_csv(criterion_rows, TABLES / 'lifting-criterion-degree-table.csv')
criterion_png = FIGURES / 'lifting-criterion-degree-divisibility.png'
fig, ax = plt.subplots(figsize=(10, 4.2))
im = ax.imshow(matrix, cmap='YlGnBu', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(degrees)))
ax.set_xticklabels(degrees)
ax.set_yticks(range(len(cover_degrees)))
ax.set_yticklabels(cover_degrees)
ax.set_xlabel('degree m of phi:S^1 -> S^1')
ax.set_ylabel('cover degree n of p_n')
ax.set_title('Lifting criterion for circle maps: lift exists exactly when n divides m')
for i, n in enumerate(cover_degrees):
    for j, m in enumerate(degrees):
        if matrix[i, j]:
            ax.text(j, i, 'yes', ha='center', va='center', fontsize=7, color='black')
fig.colorbar(im, ax=ax, ticks=[0, 1], label='lift exists')
save_matplotlib(fig, criterion_png)
plt.close(fig)

n_lift = 5
m_loop = 7
start_sheet = 0
s = np.linspace(0, 1, 300)
base_angle_turns = m_loop * s
lift_angle_turns = (start_sheet + m_loop * s) / n_lift
endpoint_sheet = int((start_sheet + m_loop) % n_lift)
path_lift_png = FIGURES / 'path-lift-endpoint-n-cover.png'
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(s, base_angle_turns, color='#6c757d', lw=2, label='base loop angle in turns')
ax.plot(s, lift_angle_turns, color='#d62828', lw=3, label='lifted angle in covering circle')
for k in range(0, m_loop + 1):
    ax.axhline(k / n_lift, color='0.9', lw=0.8, zorder=0)
ax.scatter([0, 1], [lift_angle_turns[0], lift_angle_turns[-1]], color='#d62828', s=60, zorder=4)
ax.text(1.01, lift_angle_turns[-1], f'endpoint sheet {endpoint_sheet}', va='center', fontsize=10)
ax.set_xlabel('path parameter')
ax.set_ylabel('angle measured in turns')
ax.set_title('A lifted loop records the endpoint sheet')
ax.legend(loc='upper left')
save_matplotlib(fig, path_lift_png)
plt.close(fig)

criterion_checks = {
    'criterion': 'degree m circle map lifts through p_n iff n divides m',
    'tested_cover_degrees': cover_degrees,
    'tested_map_degrees': degrees,
    'all_entries_match_mod_test': bool(all(row['lift_exists'] == (row['map_degree_m'] % row['cover_degree_n'] == 0) for row in criterion_rows)),
    'path_lift_example': {
        'cover_degree_n': n_lift,
        'base_loop_degree_m': m_loop,
        'start_sheet': start_sheet,
        'endpoint_sheet': endpoint_sheet,
        'endpoint_formula': '(start_sheet + m) mod n',
    },
}
criterion_checks_path = save_json(criterion_checks, CHECKS / 'lifting-criterion-checks.json')

assert criterion_checks['all_entries_match_mod_test']
assert endpoint_sheet == 2

display_artifact(criterion_png, width=780)
display_artifact(path_lift_png, width=760)
display_artifact(criterion_table_path)
display_artifact(criterion_checks_path)


## Monodromy as a Fiber Action

Monodromy packages all path lifting from one base point into a group action. Fix a covering `q:E -> X` and a point `x` in the base. For a fiber point `e` over `x` and a loop class `[f]` in `pi_1(X,x)`, lift the loop starting at `e`; the endpoint is another point of the same fiber. That endpoint is `e * [f]`. The monodromy theorem is what makes this well-defined on path-homotopy classes rather than on raw paths.

For the `n`-fold cover of the circle, the fiber has `n` labels. The generator of `pi_1(S^1)` acts by the cyclic permutation `j -> j+1 mod n`. This finite action contains several chapter ideas at once. It is transitive because any sheet can be reached by lifting a loop of suitable degree. The stabilizer of sheet `0` consists of loops whose degree is a multiple of `n`; that stabilizer is the induced subgroup `p_{n*}(pi_1(S^1)) = n Z`. In this example the group is abelian, so all stabilizers are the same subgroup. In nonabelian base groups the stabilizers may be conjugate without being equal, which is the reason covering isomorphism criteria are stated in terms of conjugacy classes unless base points in fibers are fixed.

The HTML artifact below is intentionally a graph rather than a geometric curve. The chapter's monodromy action is algebraic data extracted from lifting. A graph makes the action and its orbit structure explicit.


In [ ]:
monodromy_n = 6
fiber_labels = list(range(monodromy_n))
permutation = {j: (j + 1) % monodromy_n for j in fiber_labels}
action_graph = nx.DiGraph()
action_graph.add_nodes_from(fiber_labels)
action_graph.add_edges_from((j, permutation[j]) for j in fiber_labels)
orbit_from_zero = {0}
current = 0
for _ in range(monodromy_n):
    current = permutation[current]
    orbit_from_zero.add(current)
generator_order = next(k for k in range(1, monodromy_n + 2) if k % monodromy_n == 0)

angles = np.linspace(0, 2 * np.pi, monodromy_n, endpoint=False)
positions = {j: (float(np.cos(a)), float(np.sin(a))) for j, a in zip(fiber_labels, angles)}
plotly_fig = go.Figure()
for j, k in action_graph.edges():
    x0, y0 = positions[j]
    x1, y1 = positions[k]
    plotly_fig.add_trace(
        go.Scatter(
            x=[x0, x1],
            y=[y0, y1],
            mode='lines',
            line={'color': '#4361ee', 'width': 3},
            hoverinfo='skip',
            showlegend=False,
        )
    )
plotly_fig.add_trace(
    go.Scatter(
        x=[positions[j][0] for j in fiber_labels],
        y=[positions[j][1] for j in fiber_labels],
        mode='markers+text',
        marker={'size': 24, 'color': '#f4a261', 'line': {'color': '#222', 'width': 1}},
        text=[f'e{j}' for j in fiber_labels],
        textposition='middle center',
        hovertext=[f'generator sends e{j} to e{permutation[j]}' for j in fiber_labels],
        hoverinfo='text',
        showlegend=False,
    )
)
for j, k in action_graph.edges():
    x0, y0 = positions[j]
    x1, y1 = positions[k]
    plotly_fig.add_annotation(
        x=0.82 * x1 + 0.18 * x0,
        y=0.82 * y1 + 0.18 * y0,
        ax=0.68 * x1 + 0.32 * x0,
        ay=0.68 * y1 + 0.32 * y0,
        xref='x',
        yref='y',
        axref='x',
        ayref='y',
        showarrow=True,
        arrowhead=3,
        arrowsize=1.4,
        arrowwidth=2,
        arrowcolor='#4361ee',
    )
plotly_fig.update_layout(
    title='Monodromy action for a 6-sheet circle cover: generator advances one sheet',
    width=760,
    height=560,
    xaxis={'visible': False, 'scaleanchor': 'y'},
    yaxis={'visible': False},
    margin={'l': 20, 'r': 20, 't': 70, 'b': 20},
    plot_bgcolor='white',
)
monodromy_html = save_plotly_html(plotly_fig, HTML / 'monodromy-cycle-action.html')

monodromy_checks = {
    'cover_degree_n': monodromy_n,
    'fiber_labels': fiber_labels,
    'generator_permutation': [permutation[j] for j in fiber_labels],
    'orbit_from_sheet_0': sorted(int(j) for j in orbit_from_zero),
    'transitive': bool(orbit_from_zero == set(fiber_labels)),
    'stabilizer_of_sheet_0_in_Z': f'{monodromy_n}Z',
    'generator_order_on_fiber': int(generator_order),
    'strongly_connected_action_graph': bool(nx.is_strongly_connected(action_graph)),
}
monodromy_checks_path = save_json(monodromy_checks, CHECKS / 'monodromy-action-checks.json')

assert monodromy_checks['transitive']
assert monodromy_checks['generator_order_on_fiber'] == monodromy_n
assert monodromy_checks['strongly_connected_action_graph']

display_artifact(monodromy_html, width=780, height=580)
display_artifact(monodromy_checks_path)


## Proof and Invariant Scaffold

The chapter's proofs follow a useful dependency pattern. Local sections are the local inverse branches that make lifting possible. Unique lifting prevents a connected source from switching sheets after agreeing at one point. Homotopy lifting turns a path homotopy downstairs into a homotopy upstairs. Monodromy then records the endpoint of a lifted loop, and injectivity of the induced map on fundamental groups follows by lifting a null-homotopy of the projected loop.

The lifting criterion uses that machinery in the opposite direction. To build a lift of a map `phi:Y -> X`, choose paths in `Y` from a base point, lift their images, and define the lift by endpoints. The subgroup containment condition is exactly what prevents different choices of path from giving different endpoints. After that, monodromy converts topology into transitive `G`-sets: stabilizers are induced subgroups, stabilizers vary by conjugacy when the fiber point changes, and covering homomorphism or isomorphism questions become subgroup containment or equality questions. Universal covers are the final simplification: when the covering space is simply connected, its induced subgroup is trivial, so it maps to every other covering of the same base.

The dependency graph is not a replacement for proof, but it is a compact scaffold for remembering which theorem supplies which invariant. The check attached to the graph asserts that the dependency relation is acyclic; cycles in this graph would indicate a circular proof outline.


In [ ]:
proof_edges = [
    ('evenly covered neighborhoods', 'local sections'),
    ('evenly covered neighborhoods', 'unique lifting'),
    ('local sections', 'homotopy lifting'),
    ('unique lifting', 'homotopy lifting'),
    ('homotopy lifting', 'path lifting'),
    ('path lifting', 'monodromy theorem'),
    ('unique lifting', 'monodromy theorem'),
    ('monodromy theorem', 'injectivity of q_*'),
    ('path lifting', 'lifting criterion'),
    ('monodromy theorem', 'lifting criterion'),
    ('path lifting', 'monodromy action'),
    ('monodromy theorem', 'monodromy action'),
    ('monodromy action', 'induced subgroup = isotropy'),
    ('induced subgroup = isotropy', 'conjugacy theorem'),
    ('induced subgroup = isotropy', 'normal covering criterion'),
    ('lifting criterion', 'covering homomorphism criterion'),
    ('conjugacy theorem', 'covering isomorphism criterion'),
    ('covering homomorphism criterion', 'universal covering universality'),
    ('covering isomorphism criterion', 'universal covering uniqueness'),
]
proof_graph = nx.DiGraph()
proof_graph.add_edges_from(proof_edges)
proof_is_dag = nx.is_directed_acyclic_graph(proof_graph)
proof_order = list(nx.topological_sort(proof_graph))

proof_png = FIGURES / 'proof-dependency-covering-maps.png'
fig, ax = plt.subplots(figsize=(12, 8))
pos = nx.spring_layout(proof_graph, seed=11, k=1.35)
nx.draw_networkx_edges(proof_graph, pos, ax=ax, edge_color='#6c757d', arrows=True, arrowsize=15, width=1.4)
nx.draw_networkx_nodes(proof_graph, pos, ax=ax, node_color='#edf6f9', edgecolors='#006d77', node_size=1800, linewidths=1.2)
nx.draw_networkx_labels(proof_graph, pos, ax=ax, font_size=8)
ax.set_title('Proof dependency scaffold for covering maps')
ax.axis('off')
save_matplotlib(fig, proof_png)
plt.close(fig)

proof_checks = {
    'node_count': int(proof_graph.number_of_nodes()),
    'edge_count': int(proof_graph.number_of_edges()),
    'is_directed_acyclic_graph': bool(proof_is_dag),
    'topological_order': proof_order,
    'terminal_results': [node for node in proof_graph.nodes if proof_graph.out_degree(node) == 0],
}
proof_checks_path = save_json(proof_checks, CHECKS / 'proof-dependency-checks.json')

assert proof_is_dag
assert 'universal covering uniqueness' in proof_checks['terminal_results']

display_artifact(proof_png, width=850)
display_artifact(proof_checks_path)


## Universal Covers

A universal cover is a covering space that is simply connected. Once such a cover exists, it is universal in a precise sense: it covers every other covering space over the same base, and any two simply connected coverings of the same base are isomorphic as coverings. The chapter constructs universal covers for sufficiently nice spaces by using path classes from a base point as points of the new space. That construction is subtle, but the working mental model is simple: a universal cover unwraps all fundamental-group ambiguity into different points upstairs.

For `S^1`, the universal cover is `R`, and integer translations move between points in the same fiber. For the torus `T^2`, the universal cover is `R^2`, and the fiber over a base point is the integer lattice translate of any one lift. The diagram below shows a base point `(a,b)` repeated as `(a+i,b+j)` across many copies of a unit square. The projection to the torus is the coordinate-wise modulo-one map. Every marked point has the same projected image, and every path class in the torus moves a chosen lift by an integer vector.

The existence theorem in the chapter assumes local simple connectedness, then notes that a weaker semilocal simple connectedness condition is the true obstruction. Manifolds pass the local condition because small coordinate balls are simply connected. Spaces with too many essential small loops near a point can fail to have universal covers; this is why the theorem is not merely formal. The computational check here focuses on the torus model, where the universal cover and deck translations are explicit.


In [ ]:
base_point = np.array([0.23, 0.37])
translations = [(i, j) for i in range(-2, 3) for j in range(-2, 3)]
fiber_points = np.array([base_point + np.array(t) for t in translations])
projected = np.mod(fiber_points, 1.0)
projection_residual = np.max(np.abs(projected - base_point))

torus_png = FIGURES / 'universal-cover-torus-lattice.png'
fig, ax = plt.subplots(figsize=(7.5, 7.5))
for k in range(-2, 4):
    ax.axhline(k, color='0.86', lw=1)
    ax.axvline(k, color='0.86', lw=1)
ax.add_patch(plt.Rectangle((0, 0), 1, 1, fill=False, lw=3, edgecolor='#264653'))
ax.scatter(fiber_points[:, 0], fiber_points[:, 1], s=45, color='#e76f51', zorder=3, label='fiber over (a,b)')
ax.scatter([base_point[0]], [base_point[1]], s=110, color='#2a9d8f', edgecolor='black', zorder=4, label='chosen lift')
for vector, label in [((1, 0), '(1,0)'), ((0, 1), '(0,1)'), ((1, 1), '(1,1)')]:
    ax.arrow(base_point[0], base_point[1], vector[0], vector[1], length_includes_head=True, head_width=0.07, head_length=0.09, color='#4361ee', lw=2)
    ax.text(base_point[0] + vector[0] * 0.55, base_point[1] + vector[1] * 0.55, label, color='#273469', fontsize=10)
ax.set_xlim(-2.25, 3.1)
ax.set_ylim(-2.25, 3.1)
ax.set_aspect('equal')
ax.set_xlabel('x coordinate in R^2')
ax.set_ylabel('y coordinate in R^2')
ax.set_title('Universal cover R^2 -> T^2: one fiber is an integer lattice translate')
ax.legend(loc='upper left')
save_matplotlib(fig, torus_png)
plt.close(fig)

torus_checks = {
    'base_point_in_fundamental_square': [float(x) for x in base_point],
    'translation_window': {'min': -2, 'max': 2},
    'fiber_point_count_in_window': int(len(fiber_points)),
    'projection_rule': '(x,y) mod Z^2',
    'max_projection_residual': float(projection_residual),
    'deck_translation_examples': [{'translation': list(t), 'same_projected_point': bool(np.allclose(np.mod(base_point + np.array(t), 1.0), base_point))} for t in [(1, 0), (0, 1), (1, 1), (-2, 2)]],
}
torus_checks_path = save_json(torus_checks, CHECKS / 'universal-cover-torus-checks.json')

assert projection_residual < 1e-12
assert all(item['same_projected_point'] for item in torus_checks['deck_translation_examples'])

display_artifact(torus_png, width=760)
display_artifact(torus_checks_path)


## Applied Lab: Circle Covers as Subgroup Data

The circle is the smallest laboratory for the chapter's general results. Its fundamental group is `Z`, and connected coverings of the circle are controlled by subgroups of `Z`. In this notebook we do not need the full later classification theorem to use the chapter's criteria: the induced subgroup distinguishes the standard circle covers, and the lifting criterion reads directly from subgroup containment.

The table below treats finite covers `p_d:S^1 -> S^1` and the universal cover `R -> S^1`. For `d > 0`, the induced subgroup is `d Z` and the covering has `d` sheets. For the universal cover, the induced subgroup is the trivial subgroup and the fiber is countably infinite. A degree `m` map lifts through the cover exactly when `m Z` is contained in the induced subgroup. For finite `d`, that means `d | m`; for the universal cover, only degree `0` maps from the circle lift because a nonzero loop in the base would lift to a path with a different endpoint.

This lab is deliberately algebraic. The point is to practice translating a lifting problem into a subgroup test before trying to draw a complicated lift.


In [ ]:
test_degrees = list(range(-6, 7))
lab_rows = []
for d in [0, 1, 2, 3, 4, 5, 6]:
    if d == 0:
        induced = '{0}'
        sheets = 'countably infinite'
        liftable = [m for m in test_degrees if m == 0]
        covering = 'universal cover R -> S^1'
    else:
        induced = f'{d}Z'
        sheets = str(d)
        liftable = [m for m in test_degrees if m % d == 0]
        covering = f'p_{d}:S^1 -> S^1'
    lab_rows.append(
        {
            'cover_parameter_d': d,
            'covering_model': covering,
            'induced_subgroup': induced,
            'sheet_count': sheets,
            'liftable_degrees_from_-6_to_6': liftable,
        }
    )
lab_df = pd.DataFrame(lab_rows)
lab_table_path = save_csv(lab_rows, TABLES / 'circle-covering-lab.csv')

lab_checks = {
    'tested_degrees': test_degrees,
    'finite_cover_rows_match_divisibility': bool(
        all(
            row['cover_parameter_d'] == 0 or row['liftable_degrees_from_-6_to_6'] == [m for m in test_degrees if m % row['cover_parameter_d'] == 0]
            for row in lab_rows
        )
    ),
    'universal_cover_liftable_degrees': lab_rows[0]['liftable_degrees_from_-6_to_6'],
}
lab_checks_path = save_json(lab_checks, CHECKS / 'circle-covering-lab-checks.json')

assert lab_checks['finite_cover_rows_match_divisibility']
assert lab_checks['universal_cover_liftable_degrees'] == [0]

display_artifact(lab_table_path)
display_artifact(lab_checks_path)
lab_df


## Takeaways

- A covering map is not just a surjective local homeomorphism. The preimage of a small enough base neighborhood must split into complete sheets, each mapped homeomorphically onto that same neighborhood.
- Lifting is rigid. Once a lift agrees at one point over a connected source, uniqueness prevents it from changing sheet later. Homotopy lifting makes endpoints of lifted paths invariant under path homotopy.
- The lifting criterion is the chapter's main decision procedure: a lift exists exactly when the fundamental-group image of the source lands inside the subgroup induced by the cover.
- Monodromy converts path lifting into a transitive action of `pi_1(X,x)` on a fiber. Stabilizers of this action are the induced subgroups `q_*(pi_1(E,e))`.
- Changing the chosen fiber point conjugates the induced subgroup. This is why unbased covering isomorphism is naturally stated with conjugacy classes.
- A normal covering is one whose relevant induced subgroup is normal; equivalently, the monodromy stabilizers do not merely form a conjugacy class but agree in the way needed for uniform symmetry across the fiber.
- A universal cover is simply connected and covers every other covering over the same base. For `S^1` and `T^2`, the universal covers `R` and `R^2` make the fundamental group visible as integer deck translations.

## Final Sanity Checks

The last cell checks that every planned artifact exists, that files are nonempty, and that the main mathematical invariants agree with the chapter story: complete sheets for a covering, incomplete sheets for the failure example, divisibility for the lifting criterion, transitivity and stabilizer order for monodromy, acyclicity of the proof scaffold, and modulo-invariance for the universal cover of the torus.


In [ ]:
expected_artifacts = [
    storyboard_path,
    evenly_png,
    local_failure_png,
    local_checks_path,
    criterion_png,
    path_lift_png,
    criterion_table_path,
    criterion_checks_path,
    monodromy_html,
    monodromy_checks_path,
    proof_png,
    proof_checks_path,
    torus_png,
    torus_checks_path,
    lab_table_path,
    lab_checks_path,
]
assert_artifacts(expected_artifacts, min_bytes=80)

final_sanity_checks = {
    'artifact_count_checked': len(expected_artifacts),
    'all_artifacts_book_local': all(str(Path(path).resolve()).startswith(str(BOOK_ROOT.resolve())) for path in expected_artifacts),
    'evenly_covered_sheets_disjoint': local_checks['evenly_covered_model']['sheets_disjoint'],
    'evenly_covered_sheet_widths_equal': local_checks['evenly_covered_model']['equal_local_widths'],
    'local_homeomorphism_failure_incomplete_components': local_checks['restricted_exponential_failure']['incomplete_component_count'],
    'lifting_criterion_matches_mod_test': criterion_checks['all_entries_match_mod_test'],
    'path_lift_endpoint_sheet': criterion_checks['path_lift_example']['endpoint_sheet'],
    'monodromy_transitive': monodromy_checks['transitive'],
    'monodromy_generator_order': monodromy_checks['generator_order_on_fiber'],
    'proof_graph_is_dag': proof_checks['is_directed_acyclic_graph'],
    'torus_projection_residual_below_tolerance': torus_checks['max_projection_residual'] < 1e-12,
    'circle_lab_matches_divisibility': lab_checks['finite_cover_rows_match_divisibility'],
}

assert final_sanity_checks['all_artifacts_book_local']
assert final_sanity_checks['evenly_covered_sheets_disjoint']
assert final_sanity_checks['evenly_covered_sheet_widths_equal']
assert final_sanity_checks['local_homeomorphism_failure_incomplete_components'] == 2
assert final_sanity_checks['lifting_criterion_matches_mod_test']
assert final_sanity_checks['path_lift_endpoint_sheet'] == 2
assert final_sanity_checks['monodromy_transitive']
assert final_sanity_checks['monodromy_generator_order'] == monodromy_n
assert final_sanity_checks['proof_graph_is_dag']
assert final_sanity_checks['torus_projection_residual_below_tolerance']
assert final_sanity_checks['circle_lab_matches_divisibility']

final_checks_path = save_json(final_sanity_checks, CHECKS / 'final-sanity-checks.json')
assert_artifacts([final_checks_path], min_bytes=80)
display_artifact(final_checks_path)
final_sanity_checks
